# Exercise 2: Data Preparation for LLM Instruction Tuning

In this exercise, we will:
1. Load an instruction dataset
2. Format the data for instruction tuning
3. Tokenize the data
4. Create train/validation splits
5. Save processed datasets

## 1. Setup and Imports

In [ ]:
import os
import json
from pathlib import Path
import torch
from transformers import AutoTokenizer
from datasets import load_dataset, Dataset, DatasetDict
import numpy as np
import mlflow
from tqdm.auto import tqdm

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 2. Configuration

Define paths and parameters for data preparation:

In [ ]:
# Configuration
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Base model for tokenization
DATASET_NAME = "databricks/databricks-dolly-15k"  # Instruction dataset
OUTPUT_DIR = Path("./data")
MAX_LENGTH = 512  # Maximum sequence length
TRAIN_SPLIT = 0.9  # 90% train, 10% validation

# Create output directory
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print(f"Model: {MODEL_NAME}")
print(f"Dataset: {DATASET_NAME}")
print(f"Output directory: {OUTPUT_DIR}")

## 3. Load Tokenizer

Load the tokenizer for our base model.

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Set padding token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Pad token: {tokenizer.pad_token}")

## 4. Load and Explore Dataset

Load the Dolly 15K dataset and examine its structure.

In [ ]:
# Load dataset
print("Loading dataset...")
dataset = load_dataset(DATASET_NAME)

# Show dataset info
print(f"Dataset splits: {list(dataset.keys())}")
print(f"Number of examples in train: {len(dataset['train'])}")

# Examine first example
example = dataset['train'][0]
print("\nFirst example:")
for key, value in example.items():
    print(f"  {key}: {value}")

## 5. Data Formatting Function

Create a function to format the Dolly dataset for instruction tuning.
The Dolly dataset has fields: instruction, context, response, category.
We'll combine instruction and context into a prompt, and use response as the completion.

In [ ]:
def format_instruction(example):
    """Format a single example for instruction tuning."""
    instruction = example['instruction'].strip()
    context = example['context'].strip()
    response = example['response'].strip()

    # Combine instruction and context
    if context:
        prompt = f"{instruction}\n\n{context}"
    else:
        prompt = instruction

    # Format as: "### Human: {prompt}\n\n### Assistant: {response}"
    formatted = f"### Human: {prompt}\n\n### Assistant: {response}"
    
    return {"text": formatted}


## 6. Apply Formatting to Dataset

Apply the formatting function to the entire dataset.

In [ ]:
# Apply formatting
print("Formatting dataset...")
formatted_dataset = dataset['train'].map(format_instruction, remove_columns=dataset['train'].column_names)

# Check a few formatted examples
print("\nFormatted examples:")
for i in range(min(3, len(formatted_dataset))):
    print(f"Example {i+1}:\n{formatted_dataset[i]['text'][:200]}...\n")